# Visualização exploratória

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_04/03_visualizacao_exploratoria.ipynb)

![Cinco tipos de pergunta são ligados a barras, histograma e boxplot, dispersão, pontos e linha, ou barras de termos.](imagens/03_escolha_grafico.svg)

Gráficos são argumentos. Comece pela pergunta e pela escala conceitual da
variável. Toda figura terá título, eixos, unidades, descrição e tabela
equivalente. Os gráficos abaixo são calculados com Python a partir da base
fictícia; não constituem evidência histórica.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_04'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import re
from collections import Counter

import pandas as pd
import sys
from pathlib import Path
from IPython.display import display
sys.path.insert(0, str(Path.cwd())) if str(Path.cwd()) not in sys.path else None
from graficos import barras_categorias, barras_horizontais, dispersao, histograma_boxplot, serie_temporal

dados = pd.read_csv("dados/documentos.csv")

## Barras — categorias

As barras respondem a uma comparação entre categorias. Neste conjunto didático,
cada tema possui seis documentos: a igualdade foi construída deliberadamente e
não representa equilíbrio histórico. A tabela devolvida pelo código torna os
valores e o denominador auditáveis.

In [ ]:
tabela_temas = dados["tema"].value_counts().sort_index().rename("documentos").to_frame()
display(barras_categorias(
    tabela_temas.index.tolist(), tabela_temas["documentos"].tolist(),
    "Documentos por tema (n = 24)", "Tema atribuído", "Número de documentos",
    "Quatro barras de mesma altura mostram seis documentos em cada tema.",
))
tabela_temas

## Histograma e boxplot — distribuições

Se os limites dos intervalos são $b_0,b_1,\ldots,b_J$, a altura da barra $j$
é a quantidade de valores dentro daquele intervalo:

$$
h_j=\sum_{i=1}^{n}\mathbf{1}(b_{j-1}<x_i\leq b_j).
$$

Alterar os limites $b_j$ pode mudar a forma visível da distribuição, mesmo sem
alterar os documentos. O boxplot aplica a regra de 1,5 IQR: os whiskers terminam
nos valores não extremos e D023 aparece como ponto separado. A tabela informa os
intervalos e o resumo numérico equivalentes.

In [ ]:
faixas = [0, 400, 600, 800, 1000, 2200]
tabela_intervalos = dados.groupby(pd.cut(dados["palavras"], faixas), observed=False).size().rename("documentos")
display(histograma_boxplot(
    dados["palavras"].tolist(), faixas, dados["id_documento"].tolist()
))
tabela_intervalos, dados["palavras"].describe()

## Dispersão — relação entre quantitativas

Cada ponto representa um documento. Forma e cor distinguem os gêneros, enquanto
a tabela mantém os pares e IDs disponíveis. D023 deve ser inspecionado; o padrão
visual entre páginas e palavras não demonstra causalidade.

In [ ]:
display(dispersao(
    dados["paginas"].tolist(), dados["palavras"].tolist(),
    dados["genero"].tolist(), dados["id_documento"].tolist(),
    "Páginas e extensão dos documentos", "Páginas", "Palavras",
))
dados[["id_documento", "genero", "paginas", "palavras"]]

## Série temporal

Se $n_t$ documentos pertencem ao ano $t$, a média anual representada pela
linha é:

$$
\bar{x}_t=\frac{1}{n_t}\sum_{i:\,\mathrm{ano}_i=t}x_i.
$$

A fórmula torna visível o denominador anual. Mostraremos os documentos individuais
junto da média, pois cada ano contém apenas dois casos. D023 eleva a média de 1900;
a linha reflete a composição do corpus e não demonstra mudança histórica contínua.

In [ ]:
resumo_anual = dados.groupby("ano")["palavras"].agg(n="size", media="mean")
display(serie_temporal(
    dados["ano"].tolist(), dados["palavras"].tolist(), dados["id_documento"].tolist(),
    resumo_anual["media"].to_dict(), resumo_anual["n"].to_dict(),
))
resumo_anual

## Frequências textuais

Barras preservam valores melhor que nuvem de palavras. A tabela informa as
frequências e o denominador deve ser lido junto das regras de tokenização e da
lista de stopwords.

In [ ]:
tokens = re.findall(r"[a-záàâãéêíóôõúç]+", " ".join(dados["texto"]).lower())
stopwords = {"a", "o", "e", "de", "do", "da", "como", "em", "nas", "um", "uma", "também"}
tokens_conteudo = [token for token in tokens if token not in stopwords]
top = Counter(tokens_conteudo).most_common(10)
tabela_termos = pd.DataFrame(top, columns=["termo", "frequencia"])
ordenada = tabela_termos.sort_values("frequencia")
display(barras_horizontais(
    ordenada["termo"].tolist(), ordenada["frequencia"].tolist(),
    f"Termos frequentes após stopwords (Nc = {len(tokens_conteudo)})",
    "Dez barras horizontais ordenadas mostram frequências absolutas dos tokens de conteúdo.",
))
tabela_termos

## Atividade

Produza barras, histograma/boxplot, dispersão ou tempo e frequência textual. Para
cada figura, entregue tabela, descrição alternativa, escala, padrão, caso, limite
e hipótese. Explique também por que o gráfico escolhido responde à pergunta.

**Minha análise:** Escreva aqui.